# Colin Strout - Reproducible AI Assignment

### AI Statement

Models from OpenAI (GPT-4) were used to understand and review the Hugging Face Library, as well as how to export a model to ONNX format, including what format the model should be in (PyTorch, Numpy, etc.). The Container to make the application was done without the help of artifical intelligence other than runtime errors caused by the Hugging Face library used by the Container.=

In [ ]:
!pip install transformers sentencepiece torch


# **Acquire and Run a Pretrained AI Model:**

* Used Hugging Face transformer library to use the Google t5-small model as a tokenizer and generation Model

* Summarized the example technical text to compare to the outputted text from the container that uses the exported model

* Parameters used on the model were chosen to ensure consistent model performance and temporal deviation

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("t5-small", legacy=False)
model = T5ForConditionalGeneration.from_pretrained("t5-small")

text = """The RAVEN system is an internally developed battery management platform designed to monitor and optimize the performance of distributed on-site devices. It provides continuous measurement of cell voltage, temperature, and discharge rates across all units, using a lightweight telemetry protocol that minimizes bandwidth consumption. RAVEN also includes a predictive module that estimates remaining useful life based on historical load patterns and environmental conditions. The system is deployed on our local network to ensure full control over operational data and to comply with internal security requirements.
During routine operation, RAVEN identifies abnormal conditions such as rapid temperature rise, irregular voltage drift, or significant mismatch in pack balancing. When a threshold is exceeded, the system automatically triggers a notification to the maintenance dashboard and logs diagnostic details for later inspection. The diagnostic log includes recent telemetry windows, expected operating constraints, and a ranked list of probable root causes. These automated assessments help technicians prioritize field checks and reduce downtime across the device fleet.
RAVEN also supports coordinated charging schedules to prevent power spikes and extend battery longevity. The scheduler evaluates current charge levels, predicted device usage, and grid load forecasts before assigning charging windows to each unit. In addition, RAVEN periodically performs firmware consistency checks to verify that all battery controllers are running the correct version of our embedded software. Any mismatch results in an automatic quarantine of the affected device until verification is complete. This ensures reliable operation and reduces the risk of system-wide failures."""

input_text = "summarize: " + text

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=30,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)

# **Export the Pretrained AI Model to a Portable Format:**

* Used Google Colab Google Drive library to save the model in memory in the Google Drive that the Colab notebok is running in



In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

MODEL_DIR = "/content/drive/MyDrive/t5-small-summarizer"
os.makedirs(MODEL_DIR, exist_ok=True)

tokenizer.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR, safe_serialization=False)

os.listdir(MODEL_DIR)

In [ ]:
!pip install transformers onnxscript optimum[onnxruntime]

* Exported the in memory t5-small mode to ONNX format using the onnxruntime library and hugging face library to export the tokenizer

* A seperate folder was used for the ONNX format to seperate required dependencies for running the model as well as using the ONNX format (t5-small summarizer and t5-small-onnx)

* A copy of the model in onnx format and one from the saved pre-trained model in memory were created for future use when comparing the output of the models, to ensure they are the same

In [ ]:
from transformers import T5Tokenizer
import shutil
from optimum.onnxruntime import ORTModelForSeq2SeqLM
import os

ONNX_DIR  = "/content/drive/MyDrive/t5-small-onnx"
os.makedirs(ONNX_DIR, exist_ok=True)

onnx_model = ORTModelForSeq2SeqLM.from_pretrained(
    MODEL_DIR,
    export=True
)

onnx_model.save_pretrained(ONNX_DIR)
tokenizer = T5Tokenizer.from_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

* ONNX Model was loaded into the notebook from the Google Drive and ran using the same parameters as before. These parameters for the tokenizer were used for both models, as well as the temperature and length paramters for the generation of text.

* After confirming both models output the same text, the necessary files were then Downloaded to my local PC for the building of the container

In [ ]:
test_input = "summarize: " + text
inputs = tokenizer(
    test_input,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

onnx_outputs = onnx_model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=30,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

onnx_summary = tokenizer.decode(
    onnx_outputs[0],
    skip_special_tokens=True
)

print(onnx_summary)
print(summary)

# **The Following Files Were Dowloaded**
* added_tokens.json
* config.json
* decoder_model_merged.onnx (Renamed for purposed of limitng warning with library use in container)
* decoder_with_past.onnx
* encoder_model.onnx
* generation_config.json
* special_tokens_map.json
* spiece.model
* tokenizer_config.json

In [ ]:
!pip install optimum[onnxruntime]

### **Optional: Quantize the ONNX Model (INT8)**
We can use Optimum's `ORTQuantizer` to apply dynamic quantization, converting our model weights to 8-bit integers for a smaller footprint and faster CPU inference.

In [ ]:
import os
from google.colab import drive

# 1. Create a brand new, clean mount point
CUSTOM_MOUNT = '/content/mnt'
os.makedirs(CUSTOM_MOUNT, exist_ok=True)

# 2. Mount to the new location
# This avoids the conflict with the cluttered /content/drive folder
drive.mount(CUSTOM_MOUNT, force_remount=True)

# 3. Update your path
# Note: 'MyDrive' is usually a subfolder of your mount point
ONNX_DIR = os.path.join(CUSTOM_MOUNT, "MyDrive/t5-small-onnx")

if os.path.exists(ONNX_DIR):
    print(f"✅ SUCCESS! Files found: {os.listdir(ONNX_DIR)}")
else:
    print("❌ Path still not found. Listing root of Drive to check names:")
    print(os.listdir(os.path.join(CUSTOM_MOUNT, "MyDrive")))

In [ ]:
import os
import shutil
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

# 1. Verification of Paths
ONNX_DIR = "/content/mnt/MyDrive/t5-small-onnx"
QUANTIZED_DIR = "/content/mnt/MyDrive/t5-small-onnx-quantized-x86"
os.makedirs(QUANTIZED_DIR, exist_ok=True)

# 2. Pi-Specific Config (ARM64)
# dqconfig = AutoQuantizationConfig.arm64(is_static=False, per_channel=False)

# 2. x86 Specific Config for Quanization
dqconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)

# 3. The "Big Three" files required for T5 Seq2Seq
# We exclude 'model.onnx' because it's redundant/too large for the Pi Zero
target_files = [
    "encoder_model.onnx",
    "decoder_model.onnx",
    "decoder_with_past_model.onnx"
]

for file_name in target_files:
    print(f"\n--- Processing: {file_name} ---")

    # We tell the quantizer EXACTLY which file to look at
    quantizer = ORTQuantizer.from_pretrained(ONNX_DIR, file_name=file_name)

    # This will save the file as 'file_name' (no suffix) in the new folder
    quantizer.quantize(
        save_dir=QUANTIZED_DIR,
        quantization_config=dqconfig,
    )

# 4. Copy the "Glue" files (Essential for the Pi to run the model)
metadata = ["config.json", "generation_config.json", "tokenizer.json", "spiece.model"]
for f in metadata:
    src = os.path.join(ONNX_DIR, f)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(QUANTIZED_DIR, f))
        print(f"Copied metadata: {f}")

print(f"\n🚀 Success! All 3 components quantized and saved to: {QUANTIZED_DIR}")